# Layout access benchmark: dense vs full sphere — pure numpy / zarr

Companion to `layout_access.ipynb` (which uses xarray + dask). This notebook bypasses xarray entirely and reads zarr arrays eagerly into numpy. The question is whether the 24× full-sphere penalty we saw in the dask notebook is dask-graph overhead or a fundamental zarr-side cost when most chunks are empty.

If pure-numpy reads are fast for full sphere too → the penalty is dask-graph-overhead and could in principle be patched upstream. If pure-numpy is also slow → it's a zarr-side per-chunk loop and the layout choice is more constrained.

Same access patterns, same stores (`s3://xagg/bench-layout/`).

In [1]:
import gc
import os
import time

import numpy as np
import psutil
import zarr

print(f"zarr  {zarr.__version__}")
print(f"numpy {np.__version__}")

zarr  3.1.5
numpy 2.2.6


In [2]:
BASE = "s3://xagg/bench-layout"
ORDERS = [8, 10, 12]
PARENT_OFFSET = 6
S3_REGION = "us-west-2"

def store_paths(base, orders, parent_offset=6):
    paths = []
    for c in orders:
        p = max(0, c - parent_offset)
        for layout, prefix in (("dense", "dense"), ("fullsphere", "full")):
            paths.append((layout, p, c, f"{base.rstrip('/')}/{prefix}_p{p}_c{c}.zarr"))
    return paths

PATHS = store_paths(BASE, ORDERS, PARENT_OFFSET)
for layout, p, c, path in PATHS:
    print(f"{layout:11s} p={p} c={c}  {path}")

dense       p=2 c=8  s3://xagg/bench-layout/dense_p2_c8.zarr
fullsphere  p=2 c=8  s3://xagg/bench-layout/full_p2_c8.zarr
dense       p=4 c=10  s3://xagg/bench-layout/dense_p4_c10.zarr
fullsphere  p=4 c=10  s3://xagg/bench-layout/full_p4_c10.zarr
dense       p=6 c=12  s3://xagg/bench-layout/dense_p6_c12.zarr
fullsphere  p=6 c=12  s3://xagg/bench-layout/full_p6_c12.zarr


In [3]:
PROC = psutil.Process(os.getpid())

def rss_mb():
    return PROC.memory_info().rss / (1024 * 1024)

def step(label, fn):
    """Run fn(), return (label, wall_s, rss_delta_mb, return_value).
    GCs before AND after each step so RSS deltas don't leak into the next."""
    gc.collect()
    rss0 = rss_mb()
    t0 = time.perf_counter()
    val = fn()
    dt = time.perf_counter() - t0
    drss = rss_mb() - rss0
    return {"step": label, "wall_s": dt, "rss_delta_mb": drss, "value": val}

In [4]:
# Fast-read recipe: list populated chunks via the store, fetch only those
# in parallel, broadcast fill_value into the rest of the output buffer once.
# This is the upstream-PR proof-of-concept for zarr-python's empty-chunk
# enumeration cost. 1D arrays only (matches zagg's HEALPix layout).

from concurrent.futures import ThreadPoolExecutor
from zarr.core.sync import sync


def _list_populated(arr):
    """Return sorted list of populated chunk indices."""
    array_path = arr.path
    prefix = f"{array_path}/c/" if array_path else "c/"
    
    async def _list_async():
        out = []
        async for k in arr.store.list_prefix(prefix):
            out.append(k)
        return out
    
    keys = sync(_list_async())
    return sorted(
        int(k.rsplit("/", 1)[-1])
        for k in keys
        if k.rsplit("/", 1)[-1].isdigit()
    )


def fast_read_sparse(arr):
    """Eager read of a sparse 1D zarr array — list-populated-chunks-then-parallel-read.
    
    Matches `arr[:]` semantically. ~3-64× faster than `arr[:]` depending on
    sparsity, because zarr-python's slow path iterates every chunk slot in
    the chunk grid, even ones that resolve to fill_value with no I/O.
    """
    if len(arr.chunks) != 1:
        raise NotImplementedError("1D only")

    chunk_size = arr.chunks[0]
    populated_idx = _list_populated(arr)

    fill = arr.fill_value if arr.fill_value is not None else 0
    out = np.full(arr.shape, fill, dtype=arr.dtype)

    def _read_one(idx):
        return idx, arr.get_block_selection((idx,))

    with ThreadPoolExecutor(max_workers=128) as ex:
        for idx, data in ex.map(_read_one, populated_idx):
            start = idx * chunk_size
            end = min(start + chunk_size, arr.shape[0])
            out[start:end] = data[: end - start]

    return out


def fast_sel_at(arr, positions):
    """Fetch values at specified array positions, reading only the chunks
    that contain them.

    `positions` is a 1D array of integer indices in [0, arr.shape[0]).
    For N positions spanning K unique chunks, this does K parallel
    `get_block_selection` reads instead of materializing the full array.

    NOTE: This takes *array positions*, not cell IDs. For the full-sphere
    layout, position == cell_id, so passing cell IDs Just Works. For dense
    pack, cell IDs aren't positions — you'd need the layout-specific remap
    (catalog dict for dense). That's the morton-vs-block-index protocol gap
    flagged in REPORT.md §2.
    """
    if len(arr.chunks) != 1:
        raise NotImplementedError("1D only")

    chunk_size = arr.chunks[0]
    pos = np.asarray(positions).astype(np.int64)

    chunk_indices = pos // chunk_size
    target_chunks = sorted(set(int(c) for c in chunk_indices))

    def _read_one(idx):
        return idx, arr.get_block_selection((idx,))

    n_workers = min(128, len(target_chunks)) or 1
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        chunks_data = dict(ex.map(_read_one, target_chunks))

    result = np.empty(len(pos), dtype=arr.dtype)
    for j, p in enumerate(pos):
        ci = int(p) // chunk_size
        within = int(p) % chunk_size
        result[j] = chunks_data[ci][within]
    return result

In [5]:
def _open_store(path):
    """Return a zarr Store for either local path or s3://."""
    if path.startswith("s3://"):
        from zagg.store import open_store
        return open_store(path, read_only=True)
    return path

def benchmark_one(layout, parent_order, child_order, path):
    """Pure zarr/numpy access patterns."""
    metrics = []
    
    # 1) Open group (metadata only)
    def _open():
        store = _open_store(path)
        return zarr.open_group(store, path=str(child_order),
                               mode='r', zarr_format=3)
    metrics.append(step("open_group", _open))
    grp = metrics[-1]["value"]
    
    # 2) Eager read of cell_ids via fast path.
    # OLD slow-path (uncomment to compare against the empty-chunk-iteration cost):
    # def _read_cell_ids_slow():
    #     return grp["cell_ids"][:]
    # metrics.append(step("read_cell_ids", _read_cell_ids_slow))
    
    def _read_cell_ids_fast():
        return fast_read_sparse(grp["cell_ids"])
    metrics.append(step("fast_read_cell_ids", _read_cell_ids_fast))
    cell_ids = metrics[-1]["value"]
    
    # 3) Find 64 cells via np.isin on the in-memory coord — pure-numpy
    #    work over the loaded array. Cost scales with array size.
    sample_ids = cell_ids[:64].copy()
    def _isin():
        return int(np.isin(cell_ids, sample_ids).sum())
    metrics.append(step("isin_64cells", _isin))
    
    # 4) Eager read of h_mean via fast path.
    # OLD slow-path (uncomment to compare):
    # def _read_h_mean_slow():
    #     return grp["h_mean"][:]
    # metrics.append(step("read_h_mean", _read_h_mean_slow))
    
    def _read_h_mean_fast():
        return fast_read_sparse(grp["h_mean"])
    metrics.append(step("fast_read_h_mean", _read_h_mean_fast))
    h_mean = metrics[-1]["value"]
    
    # 5) Mean of the in-memory h_mean array (skipping NaN fill)
    def _mean():
        return float(np.nanmean(h_mean))
    metrics.append(step("h_mean.mean", _mean))
    
    # Drop big arrays before the sel tests so RSS readings are clean.
    del cell_ids, h_mean
    
    # 6) Targeted positional fetch — 64 cells in ONE chunk (best case).
    #    Picks the first 64 array positions from the first populated chunk.
    #    Tests: how cheap is "give me one populated chunk's worth"?
    populated = _list_populated(grp["cell_ids"])
    chunk_size = grp["cell_ids"].chunks[0]
    if populated:
        adjacent_pos = np.arange(
            populated[0] * chunk_size,
            populated[0] * chunk_size + 64,
            dtype=np.int64,
        )
        def _sel_adjacent():
            return fast_sel_at(grp["h_mean"], adjacent_pos)
        metrics.append(step("fast_sel_64adjacent", _sel_adjacent))
    
    # 7) Targeted positional fetch — 64 cells, ONE PER POPULATED CHUNK.
    #    Worst case: K = 64 unique chunks → 64 parallel GETs.
    #    Tests: how cheap is "give me cells scattered across the dataset"?
    if len(populated) >= 64:
        spread_pos = np.array([
            populated[i] * chunk_size  # first cell of each populated chunk
            for i in np.linspace(0, len(populated) - 1, 64, dtype=int)
        ], dtype=np.int64)
        def _sel_spread():
            return fast_sel_at(grp["h_mean"], spread_pos)
        metrics.append(step("fast_sel_64spread", _sel_spread))
    
    return metrics

# Sanity-check on the smallest store first
first = PATHS[0]
print(f"\nDry-run on {first[0]} c={first[2]}:")
for m in benchmark_one(*first):
    print(f"  {m['step']:20s} {m['wall_s']*1000:8.1f} ms  {m['rss_delta_mb']:+8.1f} MB")


Dry-run on dense c=8:
  open_group              601.6 ms    +106.0 MB
  fast_read_cell_ids      604.1 ms     +15.5 MB
  isin_64cells              3.8 ms      +1.6 MB
  fast_read_h_mean        269.9 ms      +1.3 MB
  h_mean.mean               4.1 ms      +1.6 MB
  fast_sel_64adjacent     100.3 ms      +0.1 MB
  fast_sel_64spread       274.0 ms      +0.5 MB


In [6]:
rows = []
for layout, p, c, path in PATHS:
    print(f"\n--- {layout} parent={p} child={c} ---")
    try:
        for m in benchmark_one(layout, p, c, path):
            rows.append({"layout": layout, "parent": p, "child": c,
                         "step": m["step"],
                         "wall_s": m["wall_s"],
                         "rss_delta_mb": m["rss_delta_mb"]})
            print(f"  {m['step']:18s} {m['wall_s']*1000:8.1f} ms  {m['rss_delta_mb']:+8.1f} MB")
    except Exception as e:
        print(f"  FAILED: {e}")
        rows.append({"layout": layout, "parent": p, "child": c,
                     "step": "ERROR", "wall_s": float('nan'),
                     "rss_delta_mb": float('nan')})


--- dense parent=2 child=8 ---
  open_group            166.6 ms      +0.0 MB
  fast_read_cell_ids    495.4 ms      +3.0 MB
  isin_64cells            2.1 ms      +0.2 MB
  fast_read_h_mean      245.7 ms      +0.5 MB
  h_mean.mean             3.8 ms      +1.6 MB
  fast_sel_64adjacent    107.9 ms      +0.0 MB
  fast_sel_64spread     261.7 ms      +0.2 MB

--- fullsphere parent=2 child=8 ---
  open_group            138.7 ms      +0.0 MB
  fast_read_cell_ids    501.5 ms      +8.1 MB
  isin_64cells           20.2 ms      +2.1 MB
  fast_read_h_mean      271.7 ms      +0.9 MB
  h_mean.mean            11.8 ms      +4.6 MB
  fast_sel_64adjacent    107.3 ms      +0.0 MB
  fast_sel_64spread     208.8 ms      +0.2 MB

--- dense parent=4 child=10 ---
  open_group            147.6 ms      +0.0 MB
  fast_read_cell_ids   4950.7 ms     +70.7 MB
  isin_64cells           25.0 ms      -7.0 MB
  fast_read_h_mean     2308.3 ms     +17.0 MB
  h_mean.mean            36.8 ms      +7.9 MB
  fast_sel_64adjacent 

In [8]:
import pandas as pd
df = pd.DataFrame(rows)

for col in ("wall_s", "rss_delta_mb"):
    print(f"\n=== {col} ===")
    pivot = df.pivot_table(
        index=["child", "step"], columns="layout", values=col,
    )
    pivot["ratio"] = pivot.get("fullsphere", float('nan')) / pivot.get("dense", float('nan'))
    print(pivot.round(3))


=== wall_s ===
layout                     dense  fullsphere    ratio
child step                                           
8     fast_read_cell_ids   0.495       0.502    1.012
      fast_read_h_mean     0.246       0.272    1.106
      fast_sel_64adjacent  0.108       0.107    0.994
      fast_sel_64spread    0.262       0.209    0.798
      h_mean.mean          0.004       0.012    3.127
      isin_64cells         0.002       0.020    9.788
      open_group           0.167       0.139    0.833
10    fast_read_cell_ids   4.951       5.112    1.033
      fast_read_h_mean     2.308       2.165    0.938
      fast_sel_64adjacent  0.109       0.122    1.117
      fast_sel_64spread    0.206       0.214    1.039
      h_mean.mean          0.037       0.067    1.814
      isin_64cells         0.025       0.136    5.438
      open_group           0.148       0.153    1.034
12    fast_read_cell_ids   5.994       5.273    0.880
      fast_read_h_mean     2.528       3.271    1.294
      fast_s

## Interpreting the result

Compare these wall-times to the dask notebook (`layout_access.ipynb`):

**If pure-numpy `read_cell_ids` is dramatically faster than dask `cell_ids.values`** for full sphere at order 12 (e.g., <30 s instead of 140 s) → the penalty is dask-graph overhead, not zarr-side. The layout decision can wait on a possible upstream patch (zarr exposing populated-chunk list, dask using it to prune).

**If pure-numpy is also slow** (similar order of magnitude to dask) → it's a zarr-side per-chunk-Python-loop cost. Even with no graph overhead, iterating 49k chunk slots takes time. Dense pack is the right call regardless of upstream patches.

**If `isin_64cells` is fast** for both (it's pure-numpy on already-loaded arrays) → confirms the penalty is at I/O / chunk-iteration, not at compute. The 200M-element numpy array is fine to operate on once it's resident.

**RSS for `read_cell_ids` at order 12**:
- dense: ~42 MB (5.3M × 8 bytes ✓)
- fullsphere: ~1.6 GB (201M × 8 bytes) — bare allocation, no graph overhead

If fullsphere RSS is ~1.6 GB rather than the ~3.0 GB we saw with dask, that confirms the extra ~1.4 GB in the dask path was task-graph overhead, not data.